# 11 · RawKernel & 커널 개념 적용 기술

> **CuPy 2일 집중 코스 — Day 2 / 단원 8 (커널 심화)**

`RawKernel`은 **CUDA C 소스 전체**를 직접 작성하고 grid/block을 직접 지정하는 가장 낮은 수준의 방법입니다.
여기서 07의 커널 개념(인덱싱·coalescing·공유메모리·occupancy)을 **CUDA C로 직접 구현**하고,
커널을 빠르게 만드는 **개념 적용(최적화) 기술**을 정리합니다.

### 왜 RawKernel을 배우는가

07에서 본 `ElementwiseKernel`/`ReductionKernel`은 CuPy가 인덱싱·타입 디스패치·리덕션 트리를 **대신
생성**해주는 편리한 상위 API입니다. 하지만 두 개 이상의 배열을 협력시키거나(스텐실처럼 이웃 원소를
함께 읽기), 블록 단위로 **공유메모리를 직접 선언**하거나, grid/block 모양 자체를 실험적으로 튜닝하려면
결국 그 아래 계층 — **CUDA C 커널 소스 자체**를 다뤄야 합니다. `RawKernel`은 CuPy가 문자열로 받은
CUDA C 코드를 **NVRTC**(런타임 컴파일러)로 즉석 컴파일해 `.cubin`으로 캐시하고, 이후 파이썬에서 일반
함수처럼 호출할 수 있게 감싸주는 얇은 래퍼입니다. 대신 그 대가로 CuPy가 자동으로 처리해주던 것들 —
브로드캐스팅, dtype 승격, 경계 검사 — 을 전부 **직접 코드로 작성**해야 합니다. 이 트레이드오프
(제어력 ↔ 편의성)를 몸으로 느끼는 것이 이 노트북의 핵심 목표입니다.

### 이 노트북의 흐름

saxpy(1D, 가장 단순한 element-wise 패턴) → 2D 스텐실(이웃 원소 접근·2D 인덱싱) → 블록 크기 스윕
(occupancy를 수치로 체감) → **최적화 기술 총정리표** → 공유메모리 블록 리덕션(가장 복잡한 협력 패턴)
→ 연습(clamp) → (심화) 공유메모리 타일 전치. 뒤로 갈수록 스레드 간 **협력의 정도**가 커지는 순서로
배치했습니다 — 협력이 없는 saxpy(각 스레드 완전히 독립) → 이웃만 읽는 스텐실 → 블록 전체가 공유메모리로
협력하는 리덕션/전치.

## 이 노트북에서 구현하는 개념 (07 참조)
- **2 인덱싱**(직접) · **5 coalescing** · **4 공유메모리(타일링/리덕션)** · **7 occupancy**(블록 튜닝)

## 학습 목표
- `RawKernel`로 1D·2D 커널을 작성하고 grid/block을 직접 지정한다.
- 공유메모리·`__syncthreads`로 **블록 리덕션**을 구현한다.
- 커널 최적화 기술(coalescing·타일링·분기 최소화·튜닝)을 적용한다.

> 본 과정 차별성('CUDA C 없이 Python만')상 RawKernel은 **심화/참고**입니다 — 같은 개념을 08~09는 Numba(Python)로 구현했습니다.

## 목차
1. [RawKernel 기초 (saxpy)](#1)
2. [2D 스텐실](#2)
3. [블록 크기 튜닝 (occupancy)](#3)
4. [커널 개념 적용(최적화) 기술](#4)
5. [공유메모리 블록 리덕션](#5)
6. [연습](#6)
7. [체크포인트](#7)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, allclose
print_env()

<a id="1"></a>
## 1. RawKernel 기초 (saxpy)

CUDA C로 `y = a*x + b` 를 작성합니다. 전역 인덱스(07의 2절)를 손으로 계산하고 경계를 검사합니다.
런치: `kernel((blocks,), (threads,), (args...))`.

### RawKernel API 뜯어보기

`cp.RawKernel(code, name)`은 두 인자를 받습니다 — CUDA C **소스 문자열 전체**와, 그 안에서 호출할
**함수 이름**(문자열)입니다. 소스 안의 함수는 반드시 두 가지 키워드를 갖춰야 합니다.
- **`extern "C"`**: CUDA 컴파일러(nvcc/NVRTC)는 기본적으로 C++이라 함수 이름을 컴파일 시 내부적으로
  변형(name mangling)합니다. `extern "C"`가 없으면 CuPy가 `saxpy`라는 이름으로 심볼을 찾지 못해
  런치가 실패합니다.
- **`__global__`**: 이 함수가 host(파이썬)에서 호출 가능한 **커널**임을 표시합니다(디바이스에서만
  호출 가능한 `__device__` 함수와 구분).

호출부 `saxpy((blocks,), (threads,), (x, y, a, b, n))`의 세 인자는 각각 **grid 크기, block 크기,
커널 인자 튜플**입니다 — 1D라도 반드시 튜플(`(blocks,)`)로 감싸야 합니다(07의 2절에서 본 grid/block
계층 그대로). 인자 튜플은 CUDA C 함수 시그니처의 타입과 **정확히 일치**해야 합니다: `float a`에는
파이썬 `float`가 아니라 `cp.float32(2.0)`을, `int n`에는 `np.int32(n)`을 명시적으로 넘겨야 합니다.
CuPy 배열(`x`, `y`)은 이미 dtype이 정해져 있어 그대로 넘기면 되지만, 스칼라는 파이썬 기본형
(`float`/`int`, 보통 64비트)과 CUDA C 타입(`float`/`int`, 32비트) 사이에 자동 변환이 없어 **타입이
틀리면 조용히 잘못된 값이 읽히거나 크래시**가 날 수 있습니다. 이는 07에서 CuPy `ElementwiseKernel`이
자동으로 처리해주던 부분을 이제부터는 직접 챙겨야 한다는 뜻입니다.

전역 인덱스 계산 `int i = blockIdx.x * blockDim.x + threadIdx.x;`는 07의 2절에서 소개한 공식을
CUDA C로 그대로 옮긴 것입니다(Numba였다면 08~09에서처럼 `cuda.grid(1)` 한 줄로 대신할 수 있었던
부분). `if (i < n)` 경계 검사는 grid가 `n`보다 큰 총 스레드 수를 만들 때(블록 크기의 배수로 반올림
하므로 보통 여유분이 생김) 남는 스레드가 배열 밖을 건드리지 않도록 막는 필수 관용구입니다.

In [ ]:
saxpy_src = r'''
extern "C" __global__
void saxpy(const float* x, float* y, float a, float b, int n){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) y[i] = a * x[i] + b;
}'''
saxpy = cp.RawKernel(saxpy_src, 'saxpy')
n = 1 << 20
x = cp.random.rand(n, dtype=cp.float32); y = cp.empty_like(x)
threads = 256; blocks = (n + threads - 1) // threads
saxpy((blocks,), (threads,), (x, y, cp.float32(2.0), cp.float32(1.0), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(2.0*cp.asnumpy(x)+1.0, y, rtol=1e-5, atol=1e-5, name='saxpy')

<a id="2"></a>
## 2. 2D 스텐실

2D 인덱싱(`blockIdx/threadIdx`의 x·y)과 경계 처리를 직접 다룹니다. grid/block을 2D 튜플로 지정.

### 스텐실(stencil) 연산이란

**스텐실**은 각 출력 원소를 계산할 때 자기 자신이 아니라 **주변(이웃) 원소들의 값을 조합**하는
패턴입니다. 유한차분법(PDE 수치해석)·이미지 컨볼루션·블러/샤프닝 필터가 모두 스텐실의 일종입니다.
여기서 쓰는 5점 스텐실(five-point stencil)은 상하좌우 4개 이웃과 자기 자신을 조합해
`y[i,j] = 0.25*(위+아래+왼쪽+오른쪽) - 자신`을 계산합니다 — 2D 라플라시안(Laplacian)의 이산 근사와
같은 형태로, `04_scipy_routines`의 2D 라플라시안/푸아송 방정식과 개념적으로 연결됩니다.

### 2D 인덱싱과 평탄화(flatten)

CUDA는 스레드/블록을 최대 3차원(`x`,`y`,`z`)까지 지원하지만, GPU 전역 메모리 자체는 **1차원 주소
공간**입니다. 그래서 커널 안에서는 `blockIdx.x/y`, `threadIdx.x/y`로 2D 논리 좌표 `(i, j)`를 얻은
뒤, `idx = j * nx + i`처럼 **행 우선(row-major)** 순서로 1차원 인덱스로 변환해 메모리에 접근합니다.
이는 NumPy 배열이 기본적으로 C-order(행 우선)로 저장되는 것과 동일한 규약이라, `x_np.ravel()`/
`xg.reshape(ny, nx)`처럼 파이썬 쪽에서도 1D ↔ 2D를 자유롭게 오갈 수 있습니다.

경계 처리(`if (i<=0 || j<=0 || i>=nx-1 || j>=ny-1) return;`)는 이웃(`idx-nx`, `idx+nx`, `idx-1`,
`idx+1`)을 읽을 때 배열 밖을 참조하지 않도록 가장자리 한 겹을 계산에서 제외하는 관용구입니다 — 1절의
`if (i<n)`과 같은 역할이지만 2D라서 네 방향을 모두 확인해야 합니다.

grid/block도 2D 튜플로 지정합니다(`block=(16,16)`, `grid=(ceil(nx/16), ceil(ny/16))`) — 07에서
배운 grid/block 계층을 그대로 두 차원으로 확장한 것뿐입니다. `idx-nx`/`idx+nx` 접근(세로 방향
이웃)은 워프 내 인접 스레드가 접근하는 주소가 아니므로 **coalescing 관점에서 완벽하지 않은데**, 이
노트북 뒤쪽(공유메모리 타일 전치)에서 같은 종류의 문제를 공유메모리로 해결하는 기법을 봅니다.

In [ ]:
stencil_src = r'''
extern "C" __global__
void stencil5(const float* x, float* y, int nx, int ny){
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    int j = blockIdx.y * blockDim.y + threadIdx.y;
    if (i <= 0 || j <= 0 || i >= nx-1 || j >= ny-1) return;
    int idx = j * nx + i;
    y[idx] = 0.25f*(x[idx-nx] + x[idx+nx] + x[idx-1] + x[idx+1]) - x[idx];
}'''
stencil5 = cp.RawKernel(stencil_src, 'stencil5')
nx, ny = 2048, 2048
x_np = np.random.rand(ny, nx).astype(np.float32)
def stencil_np(x):
    y = np.zeros_like(x)
    y[1:-1,1:-1] = 0.25*(x[:-2,1:-1]+x[2:,1:-1]+x[1:-1,:-2]+x[1:-1,2:]) - x[1:-1,1:-1]
    return y
ref = stencil_np(x_np)
xg = cp.asarray(x_np).ravel(); yg = cp.zeros_like(xg)
block = (16, 16); grid = (math.ceil(nx/16), math.ceil(ny/16))
stencil5(grid, block, (xg, yg, np.int32(nx), np.int32(ny)))
cp.cuda.Device().synchronize()
allclose(ref, yg.reshape(ny, nx), rtol=1e-5, atol=1e-5, name='stencil5')

<a id="3"></a>
## 3. 블록 크기 튜닝 (occupancy)

블록 모양에 따라 occupancy·메모리 효율이 달라져 성능이 변합니다(07의 7절). 직접 스윕해 최적점을 찾습니다.

### 실험 설계

아래에서 스윕하는 네 가지 블록 모양 `(8,8)`, `(16,16)`, `(32,8)`, `(32,16)`은 각각 블록당 스레드
수가 64 · 256 · 256 · 512개로 다르고, x축 크기(`block.x`)도 8/16/32/32로 다릅니다 — 즉 **"총
스레드 수"** 와 **"x축이 32의 배수인가"** 라는 두 변수를 동시에 바꿔가며 어느 쪽이 성능에 더 크게
기여하는지 관찰하는 실험입니다. CUDA는 블록당 스레드 수에 하드웨어 상한(보통 1024개)이 있어 이
목록 이후로도 `(32,32)`(=1024개)까지 시도해볼 여지가 있습니다. 결과 숫자와 그 이유(coalescing·
occupancy 관점)는 실행 직후 다음 셀에서 자세히 풀어봅니다.

In [ ]:
for block in [(8,8),(16,16),(32,8),(32,16)]:
    grid = (math.ceil(nx/block[0]), math.ceil(ny/block[1]))
    def run(g=grid, b=block): stencil5(g, b, (xg, yg, np.int32(nx), np.int32(ny)))
    print('block', block, '->', round(gpu_ms(bench(run, n_repeat=20, n_warmup=5)), 4), 'ms')

## 3.1 전체 격자를 타일(Tile)로 쪼개는 개념
GPU는 2048 x 2048 크기의 거대한 격자 데이터를 블록(Block)이라는 작은 타일 형태로 나눕니다.
 ``` 
[ 전체 2048 x 2048 이미지 / 행렬 ]
+------------------------------------+
| Block(0,0) | Block(1,0) | ...      |
+------------+------------+          |
| Block(0,1) | Block(1,1) |          |
|    ...     |    ...     |          |
+------------------------------------+
  <--- Grid (전체 타일들의 집합) --->
```
Grid는 전체 작업 영역이고, Block은 GPU 코어 그룹(SM)에 배정되는 독립된 타일 단위입니다. 동일한 2048 x 2048 영역이라도 타일을 어떤 모양으로 자르느냐에 따라 GPU 내부의 연산 효율이 크게 달라집니다.

- 메모리 접근 방식의 차이 (핵심 요인: Memory Coalescing)
  * 성능 차이의 가장 큰 원인은 X축 크기(block.x)가 32냐 아니냐입니다. C/CUDA 메모리는 행(Row) 방향으로 연속 저장됩니다.
  * GPU의 최소 실행 단위인 Warp(32개 스레드)가 메모리를 읽을 때의 모습을 비교해서 그려주면 가장 직관적입니다.

```
block.x = 8 인 경우 (8x8)=
Warp (32개 스레드)가 8개씩 4줄로 쪼개짐:
[Row 0: 스레드 0~7  ] -> 연속된 8개 데이터 읽음 (32B)  ───┐
[Row 1: 스레드 8~15 ] -> 떨어진 위치 8개 읽음 (32B)    ├── 4번 나누어 메모리 요쳥!
[Row 2: 스레드 16~23] -> 떨어진 위치 8개 읽음 (32B)    │  (비효율적인 띄엄띄엄 접근)
[Row 3: 스레드 24~31] -> 떨어진 위치 8개 읽음 (32B)    ───┘
```
block.x = 32 인 경우 (32x8, 32x16)
Warp (32개 스레드)가 X축으로 1줄로 길게 정렬됨:
[Row 0: 스레드 0 ~ 31 ] -> 연속된 32개 데이터(128B)를 한 번에 통째로 로드!

- 블록당 총 스레드 수와 점유율 (Occupancy)
 * 블록 하나의 총 스레드 수(block.x * block.y)가 너무 적으면 GPU 연산 장치가 노는 시간(지연 시간)을 은닉하지 못합니다.

```
[ 블록 크기별 스레드 밀도 비교 ]

 (8, 8) = 64 Threads   │ (16, 16) = 256 Threads │ (32, 16) = 512 Threads
 +-------------------+ │ +--------------------+ │ +--------------------+
 | 2 Warps           | │ | 8 Warps            | │ | 16 Warps           |
 | (일감이 부족함)   | │ | (적절함)           | │ | (GPU 자원 꽉 채움) |
 +-------------------+ │ +--------------------+ │ +--------------------+
      3.34 ms                1.90 ms                 1.42 ms (최적!)
```

- 메모리 병합 접근(Coalescing)을 위해 X축 크기를 32의 배수(block.x = 32)로 지정해야 합니다.
- 지연 시간 은닉(Latency Hiding)을 위해 블록당 총 스레드 수(block.x * block.y)를 256 ~ 512개 수준으로 넉넉히 맞춰야 합니다.

<a id="4"></a>
## 4. 커널 개념 적용(최적화) 기술

07의 개념을 실제 커널에 적용하는 대표 기술입니다.

| 기술 | 적용 개념 | 효과 |
|------|-----------|------|
| **연속 접근**(grid-stride) | 5 coalescing | 메모리 트랜잭션↓·대역폭↑ |
| **공유메모리 타일링/리덕션** | 4 메모리계층 | 전역 접근↓·재사용↑ |
| **분기 최소화** | 3 SIMT | warp divergence↓ |
| **`const`/`__restrict__`** | — | 별칭 없음 가정 → 컴파일러 최적화 |
| **블록/스레드 튜닝** | 7 occupancy | 지연 숨김 |
| **연산 융합** | — | 중간배열·커널 런치↓ |

### 기술별로 조금 더 구체적으로

- **연속 접근(coalescing) + grid-stride 루프**: 07의 5절에서 배운 대로, warp의 32개 스레드가 인접한
  주소를 읽으면 하드웨어가 이를 **한 번의 메모리 트랜잭션**(예: 128바이트 캐시라인 하나)으로 묶어
  처리합니다. grid-stride 루프(`for (; i<n; i += stride)`)는 이 coalescing 패턴을 유지하면서도,
  **grid 크기를 데이터 크기에 맞출 필요 없이** 고정된 수의 블록으로 임의 크기의 배열을 처리하게
  해주는 관용구입니다(아래 코드에서 직접 확인).
- **공유메모리 타일링/리덕션**: 전역 메모리(수백~수천 사이클 지연) 대신 블록 내부의 **공유메모리**
  (온칩, 수 사이클 지연)에 데이터를 한 번 올려두고 여러 번 재사용하는 기법입니다. 5절 공유메모리
  블록 리덕션과 (심화) 타일 전치가 이 기술의 실전 예입니다.
- **분기 최소화**: warp의 32개 스레드는 **같은 명령을 lockstep으로** 실행하므로(07의 3절 SIMT),
  `if/else`로 스레드마다 다른 경로를 타면 두 경로를 순차적으로 실행해 최악의 경우 **성능이 절반**까지
  떨어집니다(`v<lo?lo:(v>hi?hi:v)` 같은 삼항 연산자·분기 없는 수식으로 대체하면 완화됩니다 — 6절
  clamp 연습에서 사용).
- **`const`/`__restrict__`**: `__restrict__`는 "이 포인터가 가리키는 메모리는 다른 포인터와
  겹치지(alias) 않는다"는 **개발자가 컴파일러에 주는 보증**입니다. 이 보증이 없으면 컴파일러는
  `y`에 쓰는 동작이 `x`가 가리키는 값을 바꿀 수도 있다고 가정해 매번 `x`를 다시 읽는 보수적인 코드를
  생성합니다. `__restrict__`를 붙이면 컴파일러가 읽기 결과를 레지스터에 캐싱하거나 명령어 순서를
  재배치하는 등 더 공격적으로 최적화할 수 있습니다.
- **블록/스레드 튜닝**: 3절에서 실측한 것처럼 occupancy(SM에 동시에 올라간 warp 비율)는 블록 크기·
  레지스터/공유메모리 사용량에 좌우됩니다. 정답은 하드웨어·문제마다 다르므로 **직접 스윕**해서 찾는
  것이 현실적입니다.
- **연산 융합(kernel fusion)**: 여러 개의 작은 커널을 하나로 합치면 커널 런치 오버헤드(마이크로초
  단위지만 누적되면 무시할 수 없음)와 중간 결과를 전역 메모리에 썼다가 다시 읽는 왕복 비용을 없앨 수
  있습니다. `03_numpy_routines`의 `@cupy.fuse`가 이 아이디어를 자동화한 버전이고, RawKernel에서는
  애초에 여러 연산을 **한 커널 함수 안에** 직접 작성하는 것으로 같은 효과를 얻습니다.

예: grid-stride 루프로 **연속 접근 + 임의 크기 처리**를 동시에 얻습니다.

In [ ]:
# grid-stride saxpy: warp 인접 스레드가 인접 주소 접근(coalesced), 큰 n도 처리
saxpy_gs_src = r'''
extern "C" __global__
void saxpy_gs(const float* __restrict__ x, float* __restrict__ y, float a, int n){
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int stride = gridDim.x * blockDim.x;
    for (; i < n; i += stride) y[i] = a*x[i];
}'''
saxpy_gs = cp.RawKernel(saxpy_gs_src, 'saxpy_gs')
n = 1 << 24; x = cp.random.rand(n, dtype=cp.float32); y = cp.empty_like(x)
saxpy_gs((1024,), (256,), (x, y, cp.float32(3.0), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(3.0*cp.asnumpy(x), y, rtol=1e-5, atol=1e-5, name='saxpy_gs')

<a id="5"></a>
## 5. 공유메모리 블록 리덕션 (기술 종합)

합(reduction)을 **공유메모리 + `__syncthreads` + 트리 리덕션**으로 구현합니다 — 07의 개념 4·6을 RawKernel로 직접 적용.
각 블록이 부분합을 만들고, 블록 부분합을 마지막에 합칩니다. 공유메모리 크기는 런치 시 `shared_mem`으로 지정.

### 트리 리덕션(tree reduction) 동작 원리

블록 하나에 스레드가 256개 있다고 하면, 먼저 각 스레드가 전역 메모리에서 원소 하나씩을 읽어
공유메모리 배열 `sdata`에 옮겨 담습니다(이 로드는 인접 스레드가 인접 주소를 읽으므로 coalesced).
그 다음 `for (s = blockDim.x/2; s>0; s>>=1)` 루프가 매 단계마다 **활성 스레드 수를 절반으로 줄이며**
짝을 지어 더합니다: 1단계에서 128개 스레드가 `sdata[tid] += sdata[tid+128]`, 2단계에서 64개가
`+= sdata[tid+64]`, … 이런 식으로 8단계(=log₂256) 만에 256개의 값이 `sdata[0]` 하나로 모입니다.
원소 256개를 순차적으로 더하면 255번의 덧셈이 필요하지만, 트리 리덕션은 **깊이가 log₂(N)** 이라
병렬로 실행하면 8단계 만에 끝난다는 것이 핵심입니다.

**왜 매 단계마다 `__syncthreads()`가 필요한가**: 한 단계의 덧셈 결과(`sdata[tid]`)를 다음 단계에서
다른 스레드가 읽어야 하는데, 같은 블록의 스레드라도 **서로 다른 warp**에 속해 있으면 실행 속도가
정확히 같지 않습니다. `__syncthreads()`가 없으면 어떤 스레드는 아직 이전 단계 값을 쓰기 전에 다른
스레드가 그 값을 읽어가는 **데이터 레이스**가 생겨 결과가 실행마다 달라지는 미묘한 버그가 됩니다.
07의 6절에서 다룬 동기화 개념이 여기서 실전으로 쓰이는 예입니다.

### 동적 공유메모리 선언

`extern __shared__ float sdata[];`처럼 크기를 비워두면 **동적 공유메모리**가 되고, 실제 바이트 수는
런치할 때 `shared_mem=threads*4`(스레드 수 × `float` 4바이트)처럼 파이썬 쪽에서 지정합니다. 이는
커널 소스를 배열 크기(블록 크기)에 따라 다시 컴파일하지 않고도 다양한 블록 크기에 재사용할 수 있게
해주는 방식입니다.

### 왜 부분합을 한 번 더 합쳐야 하는가

커널 하나가 만들 수 있는 것은 **자기 블록 안**의 리덕션 결과까지입니다(공유메모리는 블록 경계를
넘지 못하므로). 그래서 이 예제는 블록 수만큼의 부분합 배열 `partial`을 만들고, 마지막에
`partial.sum()`으로 **두 번째 단계 리덕션**을 수행합니다 — 실전에서는 이 두 번째 합산도 커널로
(블록 1개짜리 리덕션) 처리하거나 충분히 작으면 지금처럼 CuPy의 `sum()`에 맡기는 것이 일반적입니다.
이 "블록별 부분합 → 최종 합산" 2단계 구조는 07에서 소개한 `ReductionKernel`이 내부적으로 자동
수행하는 것과 정확히 같은 패턴이며, 09의 히스토그램 역시 "블록 내 부분 히스토그램(공유메모리) →
전역 히스토그램에 병합(atomic)"이라는 같은 아이디어를 씁니다.

In [ ]:
reduce_src = r'''
extern "C" __global__
void block_sum(const float* __restrict__ x, float* partial, int n){
    extern __shared__ float sdata[];
    int tid = threadIdx.x;
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    sdata[tid] = (i < n) ? x[i] : 0.0f;   // coalesced 로드
    __syncthreads();
    for (int s = blockDim.x/2; s > 0; s >>= 1){   // 트리 리덕션
        if (tid < s) sdata[tid] += sdata[tid + s];
        __syncthreads();
    }
    if (tid == 0) partial[blockIdx.x] = sdata[0];
}'''
block_sum = cp.RawKernel(reduce_src, 'block_sum')
n = 1 << 22; x = cp.random.rand(n, dtype=cp.float32)
threads = 256; blocks = (n + threads - 1)//threads
partial = cp.empty(blocks, dtype=cp.float32)
block_sum((blocks,), (threads,), (x, partial, np.int32(n)), shared_mem=threads*4)
total = float(partial.sum())   # 블록 부분합 최종 합산
allclose(float(x.sum()), total, rtol=1e-3, atol=1e-1, name='block_sum')

<a id="6"></a>
## 6. 연습 — clamp RawKernel

`y = min(max(x, lo), hi)` 를 RawKernel(CUDA C)로 작성하세요(1D, 경계 검사·grid-stride 권장).

이 연습은 지금까지 다룬 기술을 한 번에 모아보는 문제입니다: (1) `extern "C" __global__`로 커널을
선언하고, (2) 4절의 grid-stride 루프로 임의 크기의 `n`을 고정된 grid로 처리하고, (3) `__restrict__`
힌트를 붙이고, (4) `if/else` 대신 삼항 연산자(`v<lo?lo:(v>hi?hi:v)`)로 **분기를 최소화**합니다.
07의 9절에서 같은 clamp 연산을 `ElementwiseKernel`(CuPy가 인덱싱·런치를 대신 생성)로 작성했던
것과 비교하면, RawKernel판은 인덱싱 공식과 grid/block 계산까지 전부 직접 써야 하는 대신, grid-stride
같은 세부 전략을 자유롭게 선택할 수 있다는 차이가 뚜렷하게 드러납니다.

In [ ]:
import numpy as np
import cupy as cp
import math

# (이전 실습에서 사용한 allclose 함수가 있다고 가정합니다)
def allclose(a, b, rtol=1e-5, atol=1e-5, name=''):
    np.testing.assert_allclose(a, b, rtol=rtol, atol=atol)
    print(f"[allclose OK] {name}")

# 1. CUDA C RawKernel 작성 (Grid-Stride Loop 적용)
clamp_src = r'''
extern "C" __global__
void clamp(const float* __restrict__ x, float* __restrict__ y, float lo, float hi, int n){
    int i = blockIdx.x*blockDim.x + threadIdx.x;
    int stride = gridDim.x*blockDim.x;
    for (; i < n; i += stride){ float v = x[i]; v = v<lo?lo:(v>hi?hi:v); y[i] = v; }
}'''
# RawKernel 객체 생성
clamp = cp.RawKernel(clamp_src, 'clamp')

# 2. 테스트 데이터 생성
N = 1_000_000
x_np = np.random.randn(N).astype(np.float32)
ref = np.clip(x_np, -1.0, 1.0)

# GPU 메모리 할당
d_x = cp.asarray(x_np)
d_y = cp.empty_like(d_x)

# 3. Grid 및 Block 설정
block_dim = 256
#grid_dim = math.ceil(N / block_dim)
grid_dim = 1024

# 4. 커널 런치 (스칼라 값은 np.float32, np.int32로 타입 명시 필수!)
clamp(
    (grid_dim,), 
    (block_dim,), 
    (d_x, d_y, np.float32(-1.0), np.float32(1.0), np.int32(N))
)

# GPU 작업 완료 대기
cp.cuda.Device().synchronize()

# 5. 검증 (NumPy np.clip 결과와 비교)
allclose(ref, d_y.get(), rtol=1e-5, atol=1e-5, name='clamp')

#### grid_dim 숫자:
 * 데이터가 매우 커질수록(수천만~수억 개) Grid-Stride Loop와 '고정된 Grid 크기(예: 1024)'를 조합하는 것이 성능과 안정성 면에서 유리(NVIDIA 공식 권장 패턴)합니다.

#### 데이터 크기에 맞추는 방식 (ceil(N/256))
 * 데이터가 1,000만 개라면 약 39,000개의 블록이 생성됩니다. GPU는 이 블록들을 SM(GPU 코어 그룹)에 올렸다가, 계산이 끝나면 내리고, 다음 대기 중인 블록을 다시 올리는 과정을 수없이 반복해야 합니다. 
 * 마치 "택배 1개 배달할 때마다 단기 알바생을 새로 채용하고 해고하는 것"과 같아서 관리 비용(Overhead)이 발생합니다.

#### 고정된 Grid + Stride Loop (1024 등)
 * 블록을 딱 1024개만 만듭니다. 이 블록들이 GPU 하드웨어에 한 번 쫙 깔린 뒤, 커널 내부의 for 문을 돌면서 배열의 끝까지 데이터를 쭉쭉 빨아들이며 처리합니다. 즉, "정직원들을 고용해서 할당된 구역의 
 * 택배가 끝날 때까지 계속 일하게 하는 것"입니다. 스레드를 껐다 켜는 비용이 사라집니다.

#### 하드웨어 자원의 한계 (SM 포화 상태)
 * GPU의 실제 물리적 코어(SM) 개수는 한정되어 있습니다 (예: RTX 3090은 82개, 4090은 128개).
 * 어차피 GPU가 한 번에 동시에 돌릴 수 있는 블록의 수는 하드웨어 스펙에 의해 정해져 있습니다. 
 * 한 번에 기껏해야 몇백~천여 개의 블록만 동시에 돌아가기 때문에, 굳이 Grid를 수만~수십만 개로 쪼개서 GPU 스케줄러(작업 분배기)를 피곤하게 만들 필요가 없습니다. 
 * 하드웨어를 꽉 채울 정도(예: 1024개)만 던져주고, 나머지는 C++의 고속 for 루프에 맡기는 것이 스케줄러의 부하를 줄이는 비결입니다.

<a id="tiled"></a>
## (심화) 공유메모리 타일 transpose

전치(transpose)는 **쓰기가 비연속**이라 느립니다(08 naive). **공유메모리 타일**로 읽기·쓰기를 모두 연속(coalesced)으로 만듭니다:
타일을 공유메모리에 연속으로 읽어 들이고, `__syncthreads` 후 전치해서 연속으로 씁니다. (coalescing + 공유메모리 종합)
뱅크 충돌 회피를 위해 타일 폭을 `TILE+1`로 패딩합니다.

### 왜 naive transpose의 쓰기가 비연속인가

`out[x*n+y] = a[y*n+x]`를 생각해보면, **읽기**(`a[y*n+x]`)는 같은 warp의 스레드들이 `x`만 다르므로
`y*n`은 고정, `x`가 0,1,2,...로 연속 → coalesced입니다. 그런데 **쓰기**(`out[x*n+y]`)는 같은
warp에서 `x`가 바뀌면 쓰는 주소가 `n`(예: 2048)씩 떨어진 위치로 튀어 — 완전히 흩어진(strided)
접근이 됩니다. 결국 읽기·쓰기 중 하나는 항상 비연속이 될 수밖에 없는 것이 naive transpose의
근본적인 한계입니다(2절에서 본 coalescing 개념이 정확히 여기서 문제를 일으키는 예시입니다).

### 타일링으로 두 방향 모두 연속으로 만드는 트릭

핵심 아이디어는 **"비연속 접근은 어차피 한 번은 해야 하니, 그것을 느린 전역 메모리 대신 빠른
공유메모리에서 하자"** 입니다.
1. 먼저 32×32 타일을 **전역→공유메모리로 coalesced 읽기**(`tile[threadIdx.y][threadIdx.x] =
   a[y*n+x]`, 읽기 패턴은 naive와 동일하게 연속).
2. `__syncthreads()`로 타일 전체가 공유메모리에 채워질 때까지 대기.
3. 공유메모리 안에서 **인덱스를 바꿔(전치해서) 읽고**(`tile[threadIdx.x][threadIdx.y]`), 그 결과를
   **전역메모리에 coalesced로 쓰기**(`out[ty*n+tx]`, 쓰기 인덱스도 x가 연속이 되도록 좌표를
   재배치).

즉 "뒤섞는" 작업을 전역 메모리가 아니라 **온칩 공유메모리 내부에서** 수행하도록 옮긴 것입니다.
공유메모리는 지연시간이 전역 메모리보다 수십~수백 배 낮아, 여기서 발생하는 비연속 접근의 비용이
거의 무시할 수 있는 수준이 됩니다.

### 뱅크 충돌(bank conflict)과 `TILE+1` 패딩

공유메모리는 하드웨어적으로 **32개의 뱅크(bank)**로 나뉘어 있어, 서로 다른 뱅크에 대한 접근은
동시에 처리되지만 **같은 뱅크에 여러 스레드가 동시에 접근**하면 순차 처리로 직렬화됩니다(최악의
경우 32배 느려질 수 있음). `tile[TILE][TILE]`을 `TILE=32`로 선언하면, 열(column) 방향으로 접근할
때(`tile[threadIdx.x][threadIdx.y]`처럼 두 번째 인덱스가 고정이고 첫 번째가 바뀌는 패턴) 워프의
32개 스레드가 **정확히 같은 뱅크**에 몰리는 뱅크 충돌이 발생합니다. `tile[TILE][TILE+1]`처럼 한
칸을 패딩하면 한 행의 길이가 33이 되어, 행이 바뀔 때마다 뱅크 배정이 한 칸씩 어긋나면서 32개
스레드가 서로 다른 뱅크로 흩어지게 됩니다 — 코드 한 줄(`+1`)의 아주 작은 변경이지만 효과는
상당합니다.

In [ ]:
transpose_src = r'''
#define TILE 32
extern "C" __global__
void transpose_tiled(const float* a, float* out, int n){
    __shared__ float tile[TILE][TILE+1];   // +1: 뱅크 충돌 회피
    int x = blockIdx.x*TILE + threadIdx.x;
    int y = blockIdx.y*TILE + threadIdx.y;
    if (x < n && y < n) tile[threadIdx.y][threadIdx.x] = a[y*n + x];  // 연속 읽기
    __syncthreads();
    int tx = blockIdx.y*TILE + threadIdx.x;
    int ty = blockIdx.x*TILE + threadIdx.y;
    if (tx < n && ty < n) out[ty*n + tx] = tile[threadIdx.x][threadIdx.y];  // 연속 쓰기
}'''
transpose_tiled = cp.RawKernel(transpose_src, 'transpose_tiled')
n = 2048; A = cp.random.rand(n, n, dtype=cp.float32); T = cp.empty_like(A)
block = (32, 32); grid = (n//32, n//32)
transpose_tiled(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
cp.cuda.Device().synchronize()
allclose(cp.asnumpy(A).T, T.reshape(n,n).get(), rtol=1e-5, atol=1e-5, name='tiled transpose')

**연습 — naive와 비교**: 08의 naive transpose(있으면)나 단순 버전과 타일 버전의 속도를 비교하세요(coalescing 효과).

큰 정방행렬(예: 2048×2048)에서는 두 버전의 차이가 보통 **수 배** 수준으로 뚜렷하게 나타납니다 —
차이의 크기 자체가 "coalescing이 실제 성능에 미치는 영향"을 체감하는 좋은 척도입니다. 아래 셀에서
두 커널을 같은 grid/block으로 실행해 `gpu_ms`를 직접 비교합니다.

In [ ]:
# 단순(비연속 쓰기) 비교용 RawKernel
naive_src = r'''
extern "C" __global__
void transpose_naive(const float* a, float* out, int n){
    int x = blockIdx.x*blockDim.x + threadIdx.x;
    int y = blockIdx.y*blockDim.y + threadIdx.y;
    if (x<n && y<n) out[x*n + y] = a[y*n + x];   // 쓰기 비연속
}'''
transpose_naive = cp.RawKernel(naive_src, 'transpose_naive')
def run_naive(): transpose_naive(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
def run_tiled(): transpose_tiled(grid, block, (A.ravel(), T.ravel(), np.int32(n)))
print('naive', round(gpu_ms(bench(run_naive)),4), 'ms | tiled', round(gpu_ms(bench(run_tiled)),4), 'ms')

#### 그리드(Grid) -> GPU 전체 (모든 SM에 걸침)
  * 그리드는 우리가 지시한 '전체 작업(Workload)'입니다. 
  * GPU 스케줄러는 이 그리드 안에 있는 수많은 블록들을 GPU 기판에 있는 여러 SM(Streaming Multiprocessor)들에게 쫙 뿌려줍니다.
  * 즉, 그리드는 여러 SM을 가로지르며(across) 존재합니다.


#### 블록(Block) -> 단일 SM (절대 쪼개지지 않음)
  * 가장 중요한 규칙입니다. 하나의 블록은 반드시 단 하나의 SM에만 통째로 배정됩니다.
  * 예를 들어 스레드 1024개짜리 블록을 반으로 쪼개서 절반은 1번 SM에, 절반은 2번 SM에 넣는 일은 절대 불가능합니다.
  * 그 이유는 공유 메모리(Shared Memory) 때문입니다. 
  * 같은 블록에 속한 팀원들은 이 메모리를 같이 써야 하는데, 공유 메모리는 각 SM 내부에 물리적으로 고립된 하드웨어 칩이기 때문에 다른 SM에 있는 스레드와는 메모리를 공유할 수 없습니다.


#### SM 내부 -> 여러 개의 블록 동시 수용
  * 하나의 SM은 생각보다 덩치가 큽니다. 그래서 SM 내부에 여유 공간(공유 메모리 용량, 레지스터 등)이 허락하는 한, 여러 개의 블록을 동시에 SM 내부로 받아들여서 실행할 수 있습니다.


#### 하드웨어 매핑
  * Grid = 건물 전체 (GPU)에 배달할 택배 물량 전체
  * SM (Streaming Multiprocessor) = 각 층을 담당하는 작업 부서 (물리적 하드웨어)
  * Block = 한 부서(SM)에 배정되는 1개 조 (소프트웨어적 묶음)
  * Warp = 그 부서 안에서 실제로 32명씩 줄 맞춰서 일하는 행동 대원들 (실제 실행 단위)


#### 위의 예제에서 block = (32, 32); block = (32, 8); 로 했을때의 성능 비교
- block=(32, 32). 32 x 32 = 1024. 이는 GPU가 허용하는 최대 스레드 수입니다. 
- 32 x 32 크기의 데이터 타일 위에 1024명의 스레드를 1:1로 올려두고, 각자 자기 자리의 데이터만 딱 하나씩 처리하게 만듭니다.
- 혹은 block=(32, 8). 혹은 쓰레드 1024명을 관리하는 것도 오버헤드다. 똘똘한 256명만 뽑아서 4배로 일하게 하자!"

<a id="7"></a>
## 7. 체크포인트

- [ ] RawKernel로 1D(saxpy)·2D(스텐실) 커널을 작성·런치했다
- [ ] 블록 크기 튜닝으로 occupancy 영향을 봤다
- [ ] 최적화 기술(coalescing·타일링·분기·튜닝)을 안다
- [ ] 공유메모리+`__syncthreads`로 블록 리덕션을 구현했다

다음: **`12_interop_frameworks`** — DLPack으로 PyTorch 등과 무복사 연동합니다.